# 10.4 场景微调与边缘部署 (Scenario Fine-tuning & Edge Deployment)

## 📚 本章概览 (Overview)

**学习目标**：
- 掌握 PEFT (Parameter-Efficient Fine-Tuning) 在视觉模型中的应用（LoRA for ViT）
- 理解 Few-shot 领域适应和增量类别学习策略
- 掌握模型量化（INT8/INT4）的完整流程和精度评估
- 学会将模型导出为 ONNX/TensorRT 并构建边缘推理平台

**核心问题**：通用模型在特定场景精度不够，但全量微调成本太高。如何用最少的数据和算力完成领域适配，然后把模型压缩到边缘设备上高效运行？

🏢 **业务场景**：平台同时服务三个客户——安防客户需要新增“攀爬围墙”检测（只有 80 张标注图），零售客户要将 SKU 从 200 扩展到 300 种（每个新类别 20-50 张图），医疗客户要求在边缘设备上运行且不能联网（数据隐私）。你需要为每个客户提供独立的微调方案，并在同一套基础设施上管理这些模型。

**知识地图**：本章是 Module 10 的最后一站，整合前 3 章的选型、图像和视频能力，完成从训练到部署的全链路闭环。

**预计学习时间**：4-5 小时

## 🎯 动机与背景 (Motivation)

### 为什么微调和部署要放在一起讲？

因为在实际项目中，它们从来不是独立的：
1. 微调产生的模型参数（如 LoRA 权重）直接影响部署的显存和延迟
2. 部署环境的限制（如 4GB 显存）反过来约束你可以用多大的 LoRA rank
3. 多租户场景下，微调策略决定了模型管理的复杂度

将这二者一起讨论，是为了让你建立**端到端的工程思维**——从训练决策到部署后果，每一步都是关联的。

### 要解决的实际问题

1. 只有 50-100 张标注图，如何把通用模型适配到特定场景且不过拟合？
2. INT8 量化后精度下降 5% 怎么办？什么时候值得做 QAT (Quantization-Aware Training, 量化感知训练)？
3. 多个客户的 LoRA 分支如何隔离和管理？如何在运行时无感切换？
4. 模型需要热更新——新版本上线过程中如何保证推理服务不中断？

In [ ]:
# 🔬 Micro Practice 1: LoRA for ViT
# Goal: Inject LoRA into ViT attention layers and train

import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType

# TODO: Load a pretrained ViT model from timm
# TODO: Configure LoRA (rank=8, target_modules=['qkv'])
# TODO: Train on a small custom dataset
# TODO: Compare trainable parameters vs full model

print("LoRA for ViT setup")

In [ ]:
# 🔬 Micro Practice 2: Full fine-tuning vs LoRA vs BitFit comparison
# Goal: Quantify the accuracy-efficiency trade-off of different PEFT methods

# TODO: Run 3 experiments: full FT, LoRA (r=8), BitFit
# TODO: Compare: accuracy, trainable params, training time, GPU memory
# TODO: Produce comparison table and recommendation

print("PEFT comparison setup")

In [ ]:
# 🔬 Micro Practice 3: Few-shot adaptation experiment
# Goal: Measure LoRA performance under different shot counts

# TODO: Run LoRA with 5-shot, 10-shot, 50-shot per class
# TODO: Compare accuracy and overfitting indicators
# TODO: Identify minimum viable shot count for target accuracy

print("Few-shot experiment setup")

In [ ]:
# 🔬 Micro Practice 4: Multi-LoRA branch management
# Goal: Implement runtime LoRA switching for multi-tenant serving

# TODO: Train 3 LoRA branches for 3 different domains
# TODO: Implement LoRA weight loading/unloading with LRU cache
# TODO: Measure switching latency and memory overhead

print("Multi-LoRA management setup")

In [ ]:
# 🔬 Micro Practice 5: Incremental class learning
# Goal: Add new classes without forgetting old ones

# TODO: Train base model on classes 1-10
# TODO: Add classes 11-15 via LoRA without revisiting old data
# TODO: Measure accuracy on old and new classes

print("Incremental learning setup")

In [ ]:
# 🔬 Micro Practice 6: INT8 quantization (PTQ)
# Goal: Quantize a model and evaluate accuracy loss

import torch.quantization as quant

# TODO: Apply post-training INT8 quantization
# TODO: Compare FP32 vs INT8 accuracy on validation set
# TODO: Measure speedup and model size reduction

print("INT8 quantization setup")

In [ ]:
# 🔬 Micro Practice 7: ONNX model export and optimization
# Goal: Export PyTorch model to ONNX with optimizations

import onnx
import onnxruntime as ort

# TODO: Export model to ONNX with dynamic batch size
# TODO: Apply ONNX graph optimizations (constant folding, operator fusion)
# TODO: Benchmark ONNX Runtime vs native PyTorch

print("ONNX export setup")

In [ ]:
# 🔬 Micro Practice 8: TensorRT build and benchmark
# Goal: Build TensorRT engine and compare performance

# TODO: Convert ONNX model to TensorRT engine (FP16/INT8)
# TODO: Benchmark: PyTorch vs ONNX Runtime vs TensorRT
# TODO: Analyze precision-recall impact of FP16/INT8

print("TensorRT benchmark setup")

In [ ]:
# 🔬 Micro Practice 9: FastAPI multimodal inference service
# Goal: Build a production-ready inference API with model hot-swap

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import asyncio

# TODO: Implement /infer endpoint with image upload
# TODO: Implement /admin/swap-model for hot model switching
# TODO: Add health check, metrics, and request logging

print("FastAPI inference service setup")

In [ ]:
# 🔬 Micro Practice 10: Docker containerization + load test
# Goal: Package the service and stress test under edge-like conditions

# TODO: Write Dockerfile with multi-stage build
# TODO: Run container with resource limits (--memory=4g)
# TODO: Locust/wrk load test at 100 concurrent connections
# TODO: Measure P50/P95/P99 latency and error rate

print("Docker + load test setup")

## 📖 理论基础 (Theory)

### 3.1 LoRA for Vision Transformers

LoRA (Low-Rank Adaptation, 低秩适应) 的核心假设：模型在适应新任务时，权重的变化矩阵 $\Delta W$ 是低秩的。

对于 ViT 的 Attention 层权重 $W \in \mathbb{R}^{d \times d}$，LoRA 将其分解为：

$$W' = W + \Delta W = W + BA$$

其中 $B \in \mathbb{R}^{d \times r}$，$A \in \mathbb{R}^{r \times d}$，且 $r \ll d$（通常 r=4~16）。

训练时只更新 A 和 B，冻结原始权重 W。推理时 $BA$ 可以合并到 W 中，零额外推理开销。

**ViT 中的 LoRA 注入位置**：
- Q (Query, 查询) 和 V (Value, 值) 投影矩阵是最优先的注入位置
- K (Key, 键) 投影和 MLP 层是次要候选
- Patch Embedding 层通常不注入（底层特征泛化性好）

### 3.2 量化原理

**PTQ (Post-Training Quantization, 训练后量化)**：
- 直接对训练好的 FP32 模型做量化，不需要额外训练
- 需要一个小的校准数据集（500-1000 张代表性图片）来确定激活值的动态范围

**QAT (Quantization-Aware Training, 量化感知训练)**：
- 在训练过程中模拟量化操作（fake quantization）
- 模型学会了“容忍”量化噪声，通常精度损失更小
- 成本更高（需要重新训练 1-2 epoch）

### 3.3 边缘推理的延迟模型

单帧推理总延迟 = 数据搬运 + 计算 + 同步：

$$T_{total} = T_{io} + T_{compute} + T_{sync}$$

- $T_{io}$：CPU→GPU 数据传输（量化后减少 4x，因为 INT8 vs FP32）
- $T_{compute}$：GPU 计算时间（TensorRT 优化可减少 30-50%）
- $T_{sync}$：CUDA 同步点（减少 sync 点可显著降低 P99）

## 🔨 从零实现 (Implementation from Scratch)

### NumPy 实现低秩矩阵分解和量化模拟

In [ ]:
# NumPy implementation of low-rank decomposition (LoRA core idea)
import numpy as np

def low_rank_forward(x, W, B, A, alpha=1.0):
    """
    Forward pass with LoRA: output = x @ W + (alpha/r) * x @ (B @ A)
    
    Args:
        x: input (batch, d_in)
        W: frozen weight (d_in, d_out)
        B: LoRA B matrix (d_in, r)
        A: LoRA A matrix (r, d_out)
        alpha: scaling factor
    
    Returns:
        output (batch, d_out)
    """
    r = B.shape[1]
    base = x @ W
    lora = (alpha / r) * (x @ B @ A)
    return base + lora

print("LoRA forward pass NumPy implementation")

In [ ]:
# NumPy simulation of quantization error
def simulate_quantization(weights, num_bits=8):
    """
    Simulate quantization error to understand precision loss.
    
    Args:
        weights: FP32 numpy array
        num_bits: target bit width
    
    Returns:
        quantized_weights: simulated INT weights (as float)
        quantization_error: per-element error
    """
    # TODO: Implement min-max quantization simulation
    pass

print("Quantization simulation NumPy implementation")

## ⚙️ 工程化实现 (Engineering Implementation)

### 模型管理器：多 LoRA 分支 + 热切换

In [ ]:
# Production-grade model manager with hot-swap support
import threading
from collections import OrderedDict
from typing import Dict, Optional

class ModelManager:
    """
    Manages multiple LoRA branches with runtime hot-swap.
    
    Features:
    - LRU cache for LoRA weights (configurable capacity)
    - Atomic model switching (no request interruption)
    - Version tracking and rollback capability
    - Memory-aware preloading strategy
    """
    pass

print("Model manager setup")

In [ ]:
# A/B inference router
class ABRouter:
    """
    Routes traffic between model versions with configurable split ratio.
    Supports gradual rollout (5% → 25% → 50% → 100%).
    """
    pass

print("A/B router setup")

## 🚀 综合项目 (Capstone Project)

### 项目：多租户模型微调与部署平台

**需求**：构建一个最小化的多租户模型管理平台，支持微调触发、模型管理、和推理服务。

**基础实现（必做）**：
1. 实现 LoRA 微调脚本（接收数据目录 → 输出 LoRA 权重 + 评估报告）
2. 实现 INT8 量化和 ONNX 导出
3. 构建 FastAPI 推理服务（支持 LoRA 切换）
4. Docker 化并限制资源运行

**进阶挑战（选做）**：
1. 增量类别学习——新类别注册 + 旧类别不遗忘检测
2. A/B 推理引擎——按比例分配流量到不同模型版本
3. 数据漂移监控——检测输入分布变化并触发告警

In [ ]:
# 🚀 Capstone: Multi-tenant model platform
# TODO: Implement complete fine-tuning + deployment pipeline

print("Capstone project setup")

## ❓ 常见问题与调试 (FAQ & Debugging)

### Q1: LoRA 微调后精度反而下降了？
常见原因：a) rank 太大导致过拟合小数据集（减小 rank 到 4）b) 学习率太高（LoRA 推荐 1e-3 ~ 1e-4，比全量微调高）c) alpha 设置不当（建议 alpha = 2*rank）

### Q2: INT8 量化精度断崖式下降？
分步排查：a) 校准数据集是否有代表性（覆盖各种光照/角度）b) 逐层分析找出异常层，对该层回退到 FP16 c) 如果 PTQ 不满足，切换到 QAT 训练 1-2 epoch

### Q3: ONNX 推理结果与 PyTorch 不一致？
检查：a) ONNX opset 版本 b) 输入归一化是否嵌入到模型中 c) ONNX Runtime 的 execution provider（CPU vs CUDA）

### Q4: Docker 容器内无法访问 GPU？
确认：a) 安装了 nvidia-container-toolkit b) docker run 加了 --gpus all c) 基础镜像包含 CUDA runtime

### Q5: 模型热切换时推理延迟抖动？
预加载策略：a) 将所有 LoRA 权重加载到 GPU 显存，通过指针切换 b) 如果显存不够，用 background thread 预加载下一个模型 c) 切换期间用旧模型继续服务，切换完成后原子替换

### Q6: 边缘设备显存不够同时跑检测+分类？
串行化策略：a) 先跑轻量检测（筛选 ROI）b) 仅在检测到目标时触发分类 c) 分类模型可以常驻显存（LoRA 分支小）或 loaded on demand

## 📝 总结与展望 (Summary)

### 核心要点回顾
1. LoRA for ViT 能以 < 2% 的参数量实现接近全量微调的效果，是多租户场景的最优解
2. PTQ 量化 + ONNX Runtime 是边缘部署的标准组合，通常可实现 2-4x 加速
3. 模型热切换不是炫技而是刚需——生产环境中模型升级不应中断服务
4. 微调策略和部署方案是联动的：LoRA rank 的选择影响显存，量化精度损失影响微调是否需要补偿

### Module 10 全模块回顾

```
10.1 选型 → 知道用什么模型
10.2 图像 → 让模型看懂图片
10.3 视频 → 让模型理解发生了什么
10.4 微调+部署 → 适配具体场景并跑在边缘设备上
```

你已经从零开始，完整走通了「多模态小模型平台」的全链路。

### 💡 思考题
1. 如果 10 个客户各需要不同的 LoRA 分支，显存不够同时加载怎么办？设计一个缓存策略。
2. 量化模型的精度损失在不同类别上的分布是否均匀？某类损失显著更大说明什么？
3. 你的平台上线 6 个月后，出现了新的 SOTA 轻量模型（比当前模型精度高 3%、速度快 20%）。如何评估切换的价值和成本？切换流程如何设计？

### 下一步
你已经完成了 Module 10 的学习！建议：
- 选择一个实践项目（安防/零售/医疗）做深度拓展
- 将 Module 10 的知识与 M08 (Agent/RAG) 结合——比如用 Agent 编排多模态分析流程
- 关注 CLIP 后续工作 (SigLIP, Alpha-CLIP) 和 TinyViT V2 等最新进展